# Is the exposure classifier of `kappa_exposure_classes.rs` what its docs say?

**Question.** The 2026-10-20 experiment classified boundary cells by exposure u (unvisited 4-neighbours; "flat" u = 1,
"corners, bays, tips" u >= 2). Does the instrument compute exactly that, consistently with `src/probe.rs`?
Links: `README.md` (registered P1-P7, N1-N5, results), `NOTEBOOK.md`, `scripts/check_classifier.py` (z3 checker),
`../2026-10-kappa-exposure-classes/`, `notes/proof-track-2026-10-20.md`.

**Assumptions.** The spec is taken from the docs, not the code. The harness runs a verbatim copy of the instrument's
`Region`/`run` (asserted at run time). Deterministic: no random seeds except the harness's fixed splitmix walk seeds.

## Provenance

In [1]:
import csv, hashlib, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments/archive/2026-10-exposure-classifier-check"
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip())
for p in ["results/table.csv", "results/walks.csv", "results/paths.csv", "results/dump.log", "results/check.txt", "scripts/check_classifier.py"]:
    print(p, hashlib.sha256((EXP / p).read_bytes()).hexdigest())
print((EXP / "results/dump.log").read_text())

commit 99d0a3c76017c544d162ff07ee8e71c1d983bb69
results/table.csv f9e0ae566c552685b9096889433a5c7dd3ec090985a88d4e6e22c386963bcdeb
results/walks.csv b1d9cc6f6d2cac0a5ae14774b170de88ee463254f43f8568b555c992084ca99a
results/paths.csv fe9a58da10ec2a55174d905f32e690bd74f2af9a35f519233e27a12bfe42ad7f
results/dump.log db2b6b0687104959b35328c5000062b43e8b75d98c42df3778293307ed9488a3
results/check.txt 2b189f1f5fdd6a30cfd32c0cf4fc193235673428d20f9b8298b4777d2bd9bdd7
scripts/check_classifier.py f87cd850c70f52dca641b91b5e8138fedfe2cadbfee1dfe78488887a154c8a78
date_utc 2026-10-25T09:16:43Z
commit 52bcd60fc8051e7f953659919dc2e4c4394e7788
dirty_src_examples 0
rustc rustc 1.98.1 (48a229cea 2026-09-01)
binary_sha256 bade93733daa6d66bd1bafa227073dbc3a4788139c5fe1cc7445399970875dc4
instrument_sha256 5ad6e2f387ce9861cd6fabfef1c806ecfbce20834e5058c97ae337a413a12cb3
harness_main_sha256 2d86ebaad29da73a765ff8f9c04a16f7bacd6cb05fe0d5da86b6af35cb681c19
harness_body_sha256 847c1ad2e9204639e6fc5090044cf8b2f52c0

## Checker verdicts (from `results/check.txt`)

In [2]:
lines = (EXP / "results/check.txt").read_text().splitlines()
verdicts = [l for l in lines if l.startswith(("ok", "FAIL"))]
print(len(verdicts), "checks;", sum(l.startswith("ok") for l in verdicts), "as expected")
for l in verdicts:
    if " N" in l[:8] or l[5:8] in ("P5 ", "P7 ", "P4b"):
        print(l)
assert lines[-1].strip() == "65/65 checks as expected" 

65 checks; 65 as expected
ok   P4b replay == instrument == probe on 224 walks, 576 windows              0 mismatches
ok   P5  visit(i): invariant & pc bookkeeping preserved, no u8 underflow      unsat (proved)
ok   P7  scripted walks: p4 = b4 = f4 = 0 in every emitted window             0 nonzero
ok   P7  2026-10-20 classes.csv: p4 = b4 = f4 = 0 in all 6154 rows            0 nonzero; k range 9..26
ok   N1  table with class clipped at 2 == spec?                               sat (counterexample/witness)
ok   N2  table with one flipped entry (mask 511 centre 0->1) == spec?         sat (counterexample/witness)
ok   N3  walk row with b2 + 1 rejected by replay                              1 mismatch
ok   N4  asymmetric spec (N, E, S only) D4-equivariant?                       sat (counterexample/witness)
ok   N5  Moore-count spec (8 nbrs) == Rust table?                             sat (counterexample/witness)
ok   N6  mutant visit() (E neighbour not decremented) preserves invariant?    sat 

## Fast re-check of the truth table (pure Python, independent of z3)

In [3]:
CELLS = [(w % 3 - 1, w // 3 - 1) for w in range(9)]
NBR = [(0, -1), (1, 0), (0, 1), (-1, 0)]
rows = list(csv.DictReader((EXP / "results/table.csv").open()))
bad = 0
for r in rows:
    m = int(r["mask"]); V = {c for w, c in enumerate(CELLS) if m >> w & 1}
    for w, (x, y) in enumerate(CELLS):
        exp = sum((x + dx, y + dy) not in V for dx, dy in NBR) if (x, y) in V else 255
        bad += int(r[f"u{w}"]) != exp
print("table entries disagreeing with spec:", bad, "of", 512 * 9)
dist = {}
for r in rows:
    if r["u4"] != "255": dist[int(r["u4"])] = dist.get(int(r["u4"]), 0) + 1
print("centre class over visited-centre patterns:", dict(sorted(dist.items())))

table entries disagreeing with spec: 0 of 4608
centre class over visited-centre patterns: {0: 16, 1: 64, 2: 96, 3: 64, 4: 16}


## Conclusion

P1-P7 held, N1-N6 were rejected (README Results). The 2026-10-20 P3/P4 verdicts stand; the class labels are exposure
counts, not shapes: u = 2 includes one-cell strips, u = 1 includes T-junction cells, and u = 4 never occurs after the
first step. **Next:** none required for this instrument; a shape-resolved split of u = 2 (corner vs strip) would be a
new registered measurement.